# 📡 RabbitMQ 消息队列交互式教程

欢迎学习RabbitMQ消息队列！本教程将带你从零开始掌握RabbitMQ的核心概念和实际应用。

## 🌐 学习目标

- 理解消息队列的基本概念和优势
- 掌握RabbitMQ的安装和连接方法
- 学会创建队列、发送和接收消息
- 理解交换机(Exchange)的不同类型
- 掌握路由(Routing)和绑定(Binding)
- 实现一个完整的消息队列应用

---

## 📅 目录

1. [消息队列简介](#1-消息队列简介)
2. [环境准备](#2-环境准备)
3. [连接RabbitMQ](#3-连接rabbitmq)
4. [Hello World - 第一个消息](#4-hello-world---第一个消息)
5. [工作队列](#5-工作队列)
6. [交换机详解](#6-交换机详解)
7. [路由和模式匹配](#7-路由和模式匹配)
8. [实践项目](#8-实践项目)
9. [最佳实践](#9-最佳实践)

---

<a id='1-消息队列简介'></a>
## 1. 📌 消息队列简介

### 💡 什么是消息队列？

消息队列(Message Queue)是一种进程间通信或同一进程的不同线程间的通信方式。

### 🤔 为什么要使用消息队列？

1. **🔄 解耦** - 将生产者和消费者分离
2. **⏳ 异步** - 非阻塞处理，提高系统响应速度
3. **📊 削峰填谷** - 缓解高并发压力
4. **🔗 可靠性** - 消息持久化，确保不丢失

### 📋 RabbitMQ 核心概念

```
Producer 🎉🔍 Exchange 📦 Queue 📦 Consumer
          💰 Bindings       💰 Bindings
```

| 概念 | 说明 |
|------|------|
| **Producer** | 消息生产者，负责发送消息 |
| **Consumer** | 消息消费者，负责接收消息 |
| **Queue** | 队列，存储消息的缓冲区 |
| **Exchange** | 交换机，决定消息如何路由到队列 |
| **Binding** | 绑定，连接交换机和队列的规则 |
| **Routing Key** | 路由键，消息的属性决定路由方式 |

---

<a id='2-环境准备'></a>
## 2. 🏠 环境准备

首先，安装必要的Python库：

In [ ]:
# 安装pika库 (RabbitMQ的Python客户端)
!pip install pika -q

# 检查安装状态
import sys
print("🎉 pika库安装完成!")
print(f"Python版本: {sys.version}")

### ⚙️ RabbitMQ Docker部署 (可选)

如果你本地没有RabbitMQ服务，可以使用Docker快速启动：

In [ ]:
# 如果你有Docker，可以运行以下命令启动RabbitMQ
# !docker run -d --name rabbitmq -p 5672:5672 -p 15672:15672 rabbitmq:management

# 默认账户: guest/guest
# Web管理接口: http://localhost:15672

print("💡 如果你有Docker，可以运行上面的命令启动RabbitMQ")
print("💰 默认连接参数: host='localhost', port=5672, user='guest', password='guest'")

---

<a id='3-连接RabbitMQ'></a>
## 3. 🔌 连接RabbitMQ

让我们创建连接管理类，方便后续使用：

In [ ]:
import pika
from typing import Optional

class RabbitMQConnection:
    """
    RabbitMQ连接管理类
    """
    
    def __init__(
        self,
        host: str = 'localhost',
        port: int = 5672,
        username: str = 'guest',
        password: str = 'guest',
        virtual_host: str = '/'
    ):
        self.host = host
        self.port = port
        self.username = username
        self.password = password
        self.virtual_host = virtual_host
        self._connection: Optional[pika.BlockingConnection] = None
        self._channel: Optional[pika.channel.Channel] = None
    
    def connect(self) -> pika.channel.Channel:
        """建立连接"""
        credentials = pika.PlainCredentials(self.username, self.password)
        parameters = pika.ConnectionParameters(
            host=self.host,
            port=self.port,
            virtual_host=self.virtual_host,
            credentials=credentials
        )
        
        self._connection = pika.BlockingConnection(parameters)
        self._channel = self._connection.channel()
        
        print(f"🟡 成功连接到 RabbitMQ!")
        print(f"   📅 主机: {self.host}:{self.port}")
        print(f"   👤 用户: {self.username}")
        
        return self._channel
    
    def close(self):
        """关闭连接"""
        if self._connection and self._connection.is_open:
            self._connection.close()
            print("🔍 连接已关闭")
    
    @property
    def channel(self) -> pika.channel.Channel:
        """获取信道"""
        if self._channel is None:
            return self.connect()
        return self._channel

# 测试连接
print("=" * 50)
print("📢 测试RabbitMQ连接类")
print("=" * 50)

try:
    rabbitmq = RabbitMQConnection()
    channel = rabbitmq.connect()
    rabbitmq.close()
    print("🏆 连接测试成功!")
except Exception as e:
    print(f"❌ 连接失败: {e}")
    print("💡 提示: 请确保RabbitMQ服务已启动")

---

<a id='4-Hello World - 第一个消息'></a>
## 4. 👋 Hello World - 第一个消息

让我们从最简单的场景开始：发送和接收一条消息。

### 📦 消息生产者 (Producer)

In [ ]:
def send_message(message: str, queue_name: str = 'hello'):
    """
    发送消息到指定队列
    
    Args:
        message: 要发送的消息内容
        queue_name: 队列名称
    """
    # 创建连接
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    # 声明队列 (如果不存在会自动创建)
    channel.queue_declare(queue=queue_name, durable=True)
    
    # 发送消息
    channel.basic_publish(
        exchange='',  # 使用默认交换机
        routing_key=queue_name,  # 路由键即队列名
        body=message.encode('utf-8'),  # 消息内容
        properties=pika.BasicProperties(
            delivery_mode=2,  # 消息持久化
        )
    )
    
    print(f"📨 已发送: '{message}' 到队列 '{queue_name}'")
    
    connection.close()

# 测试发送消息
send_message("Hello RabbitMQ!")

### 📩 消息消费者 (Consumer)

In [ ]:
def receive_message(queue_name: str = 'hello', auto_ack: bool = False):
    """
    从指定队列接收消息
    
    Args:
        queue_name: 队列名称
        auto_ack: 是否自动确认
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    # 声明队列
    channel.queue_declare(queue=queue_name, durable=True)
    
    def callback(ch, method, properties, body):
        """消息回调函数
        
        Args:
            ch: 信道
            method: 配送信息
            properties: 消息属性
            body: 消息内容
        """
        message = body.decode('utf-8')
        print(f"📩 接收到: '{message}'")
        
        if not auto_ack:
            # 手动确认消息
            ch.basic_ack(delivery_tag=method.delivery_tag)
    
    # 设置消息接收
    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=auto_ack
    )
    
    print(f"🔍 正在等待接收消息 from '{queue_name}'...")
    print("   按 Ctrl+C 终止")
    
    channel.start_consuming()

# 注意：这个函数会随时间等待，请先发送消息再运行
# receive_message('hello')

### 🎥 跨过时间巡检队列

让我们检查刚才发送的消息是否在队列中：

In [ ]:
def check_queue(queue_name: str = 'hello'):
    """检查队列状态"""
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    # 获取队列信息
    result = channel.queue_declare(queue=queue_name, durable=True, passive=True)
    message_count = result.method.message_count
    consumer_count = result.method.consumer_count
    
    print(f"📊 '{queue_name}' 队列状态:")
    print(f"   - 消息数量: {message_count}")
    print(f"   - 消费者数量: {consumer_count}")
    
    connection.close()
    
    return message_count, consumer_count

# 检查队列
check_queue('hello')

---

<a id='5-工作队列'></a>
## 5. 📋 工作队列 (Work Queue)

工作队列用于分发耗时的任务给多个worker处理。

### 💡 特性

1. **🔼 负载均衡** - 任务分配给多个消费者
2. **🔄 缓存** - 消息会等待直到有空闲的worker
3. **✨ 劳动者轮询(Round-Robin)** - 每个消息只被处理一次

In [ ]:
import time
import random

def send_task(task_message: str, queue_name: str = 'task_queue'):
    """
    发送任务到工作队列
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    # 声明队列 (存储任务)
    channel.queue_declare(queue=queue_name, durable=True)
    
    # 消息分配
    channel.basic_publish(
        exchange='',
        routing_key=queue_name,
        body=task_message.encode('utf-8'),
        properties=pika.BasicProperties(
            delivery_mode=2,  # 持久化
        )
    )
    
    print(f"📨 任务已发送: {task_message}")
    connection.close()

def worker(worker_id: int, queue_name: str = 'task_queue'):
    """
    工作者函数
    
    Args:
        worker_id: 工作者标识
        queue_name: 队列名称
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    channel.queue_declare(queue=queue_name, durable=True)
    
    # 每次只处理一个消息
    channel.basic_qos(prefetch_count=1)
    
    def callback(ch, method, properties, body):
        task = body.decode('utf-8')
        dots = '.' * len(task)
        print(f"👷‍♂️ Worker-{worker_id}: 开始处理 {task}")
        
        # 模拟处理时间 (每个字符做1秒)
        time.sleep(len(task))
        
        print(f"✅ Worker-{worker_id}: 完成 {task}{dots}")
        ch.basic_ack(delivery_tag=method.delivery_tag)
    
    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback
    )
    
    print(f"👷‍♂️ Worker-{worker_id} 已启动, 正在等待任务...")
    channel.start_consuming()

# 测试: 发送多个任务
print("=" * 50)
print("📢 发送测试任务")
print("=" * 50)

tasks = [
    "Easy task",
    "Medium task...",
    "Hard task.......",
]

for task in tasks:
    send_task(task)

print(f"
🎯 共发送 {len(tasks)} 个任务")

---

<a id='6-交换机详解'></a>
## 6. 📦 交换机详解

交换机是RabbitMQ的核心组件，负责接收消息并将其路由到队列。

### 📅 四种交换机类型

| 类型 | 算法 | 说明 |
|------|------|------|
| **direct** | 精确匹配 | 完全匹配路由键 |
| **fanout** | 广播 | 发送到所有绑定队列 |
| **topic** | 主题匹配 | 支持通配符匹配 |
| **headers** | 头部匹配 | 根据头部属性匹配 |

### 🔹 direct 交换机

In [ ]:
def demo_direct_exchange():
    """
    direct交换机示例
    
    消息根据路由键被路由到相应的队列
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    exchange_name = 'direct_exchange'
    
    # 声明交换机
    channel.exchange_declare(
        exchange=exchange_name,
        exchange_type='direct'
    )
    
    # 声明多个队列
    queues = ['info_queue', 'warning_queue', 'error_queue']
    routing_keys = ['info', 'warning', 'error']
    
    for queue, routing_key in zip(queues, routing_keys):
        channel.queue_declare(queue=queue, durable=True)
        # 绑定交换机和队列
        channel.queue_bind(
            exchange=exchange_name,
            queue=queue,
            routing_key=routing_key
        )
        print(f"🔎 {queue} 绑定到 {exchange_name} 令牌={routing_key}")
    
    # 发送消息
    messages = [
        ('info', 'This is an info message'),
        ('warning', 'This is a warning message'),
        ('error', 'This is an error message'),
    ]
    
    for routing_key, message in messages:
        channel.basic_publish(
            exchange=exchange_name,
            routing_key=routing_key,
            body=message.encode('utf-8')
        )
        print(f"📨 发送 [{routing_key}]: {message}")
    
    connection.close()
    print("
🏆 direct交换机测试完成!")

demo_direct_exchange()

### 🟠 fanout 交换机

In [ ]:
def demo_fanout_exchange():
    """
    fanout交换机示例
    
    所有绑定的队列都会接收到消息 (广播)
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    exchange_name = 'fanout_exchange'
    
    # 声明交换机
    channel.exchange_declare(
        exchange=exchange_name,
        exchange_type='fanout'
    )
    
    # 创建两个队列
    queue1 = 'subscriber_1_queue'
    queue2 = 'subscriber_2_queue'
    
    channel.queue_declare(queue=queue1, durable=True)
    channel.queue_declare(queue=queue2, durable=True)
    
    # 绑定到交换机 (无需路由键)
    channel.queue_bind(exchange=exchange_name, queue=queue1)
    channel.queue_bind(exchange=exchange_name, queue=queue2)
    
    print(f"🔎 已将 {queue1} 和 {queue2} 绑定到 fanout 交换机")
    
    # 发送一条消息
    message = "警报：所有订阅者都将接收到此消息!"
    channel.basic_publish(
        exchange=exchange_name,
        routing_key='',  # fanout无需路由键
        body=message.encode('utf-8')
    )
    
    print(f"📨 已发送: {message}")
    
    connection.close()
    print("🏆 fanout交换机测试完成!")
    print(f"💡 {queue1} 和 {queue2} 都将接收到这条消息")

demo_fanout_exchange()

### 🔵 topic 交换机

In [ ]:
def demo_topic_exchange():
    """
    topic交换机示例
    
    支持通配符:
    - * 可以替代一个单词
    - # 可以替代零或多个单词
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    exchange_name = 'topic_exchange'
    
    # 声明交换机
    channel.exchange_declare(
        exchange=exchange_name,
        exchange_type='topic'
    )
    
    # 定义绑定规则
    bindings = [
        ('*.orange.*', 'orange_queue'),      # 任何包含 orange 的三部分消息
        ('*.*.rabbit', 'rabbit_queue'),      # 任何以 rabbit 结尾的消息
        ('lazy.#', 'lazy_queue'),           # 以 lazy 开头的所有消息
    ]
    
    for pattern, queue in bindings:
        channel.queue_declare(queue=queue, durable=True)
        channel.queue_bind(
            exchange=exchange_name,
            queue=queue,
            routing_key=pattern
        )
        print(f"🔎 {queue} 绑定规则: {pattern}")
    
    print("\n📨 发送测试消息:")
    
    # 测试消息
    test_messages = [
        'quick.orange.rabbit',
        'lazy.orange.elephant',
        'quick.orange.fox',
        'lazy.brown.fox',
        'lazy.pink.rabbit',
    ]
    
    for routing_key in test_messages:
        channel.basic_publish(
            exchange=exchange_name,
            routing_key=routing_key,
            body=f"[测试] {routing_key}".encode('utf-8')
        )
        print(f"   → {routing_key}")
    
    connection.close()
    print("\n🏆 topic交换机测试完成!")
    print("\n💡 包含 'orange' 的三部分消息 → orange_queue")
    print("💡 以 'rabbit' 结尾的消息 → rabbit_queue")
    print("💡 以 'lazy' 开头的消息 → lazy_queue")

demo_topic_exchange()

---

<a id='7-路由和模式匹配'></a>
## 7. 🔎 路由和模式匹配

让我们更深入地理解路由机制。

In [ ]:
# 绑定表（存储在内存中）
binding_rules = {
    'direct': {
        'exchange': 'direct_exchange',
        'bindings': [
            {'queue': 'info_queue', 'routing_key': 'info'},
            {'queue': 'warning_queue', 'routing_key': 'warning'},
            {'queue': 'error_queue', 'routing_key': 'error'},
        ]
    },
    'fanout': {
        'exchange': 'fanout_exchange',
        'bindings': [
            {'queue': 'subscriber_1_queue', 'routing_key': ''},
            {'queue': 'subscriber_2_queue', 'routing_key': ''},
        ]
    },
    'topic': {
        'exchange': 'topic_exchange',
        'bindings': [
            {'queue': 'orange_queue', 'routing_key': '*.orange.*'},
            {'queue': 'rabbit_queue', 'routing_key': '*.*.rabbit'},
            {'queue': 'lazy_queue', 'routing_key': 'lazy.#'},
        ]
    }
}

def visualize_routing():
    """
    可视化路由机制
    """
    print("=" * 60)
    print("📊 RabbitMQ 路由机制视图")
    print("=" * 60)
    
    for exchange_type, config in binding_rules.items():
        print(f"\n📦 [{exchange_type.upper()} EXCHANGE]")
        print(f"   \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
        
        for binding in config['bindings']:
            routing_key = binding['routing_key'] or '(empty)'
            print(f"   \u2502     \u2192 [{routing_key}] \u2500\u2500\u2500\u2500\u2500\u2500> {binding['queue']}")
        print()

visualize_routing()

---

<a id='8-实践项目'></a>
## 8. 🎯 实践项目：网购系统消息处理

让我们构建一个完整的购物系统消息队列应用。

In [ ]:
import json
import time
from datetime import datetime

class ShoppingSystem:
    """
    销售系统消息管理类
    """
    
    def __init__(self):
        self.connection = pika.BlockingConnection(
            pika.ConnectionParameters(host='localhost')
        )
        self.channel = self.connection.channel()
        self.exchange_name = 'shopping_exchange'
        self._setup_exchange_and_queues()
    
    def _setup_exchange_and_queues(self):
        """设置交换机和队列
        """
        # 声明主题交换机
        self.channel.exchange_declare(
            exchange=self.exchange_name,
            exchange_type='topic'
        )
        
        # 定义队列
        queues_config = {
            'order_processing': ['order.created', 'order.updated'],
            'inventory': ['product.*'],
            'notification': ['order.*', 'payment.*'],
            'analytics': ['#'],  # 接收所有
        }
        
        for queue, patterns in queues_config.items():
            self.channel.queue_declare(queue=queue, durable=True)
            for pattern in patterns:
                self.channel.queue_bind(
                    exchange=self.exchange_name,
                    queue=queue,
                    routing_key=pattern
                )
    
    def publish_event(self, event_type: str, data: dict):
        """
        发布事件
        
        Args:
            event_type: 事件类型 (如: order.created, payment.success)
            data: 事件数据
        """
        message = {
            'event_type': event_type,
            'data': data,
            'timestamp': datetime.now().isoformat()
        }
        
        self.channel.basic_publish(
            exchange=self.exchange_name,
            routing_key=event_type,
            body=json.dumps(message).encode('utf-8'),
            properties=pika.BasicProperties(
                delivery_mode=2,
                content_type='application/json'
            )
        )
        
        print(f"📨 [{event_type}] 已发布: {data}")
    
    def close(self):
        """关闭连接
        """
        self.connection.close()

# 测试销售系统
print("=" * 60)
print("💳 销售系统消息处理测试")
print("=" * 60)

shopping = ShoppingSystem()

# 模拟发生的事件
events = [
    ('order.created', {'order_id': 'ORD001', 'customer': '张三', 'amount': 299.99}),
    ('payment.success', {'order_id': 'ORD001', 'payment_method': 'credit_card'}),
    ('product.stock_update', {'product_id': 'PROD123', 'quantity': -1}),
    ('order.shipped', {'order_id': 'ORD001', 'tracking': 'SF123456'}),
]

for event_type, data in events:
    shopping.publish_event(event_type, data)

shopping.close()
print("\n🏆 销售系统测试完成!")

### 👷‍♀️ 消费者实现

In [ ]:
def shopping_consumer(queue_name: str):
    """
    销售系统消费者
    """
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host='localhost')
    )
    channel = connection.channel()
    
    channel.queue_declare(queue=queue_name, durable=True)
    channel.basic_qos(prefetch_count=1)
    
    def callback(ch, method, properties, body):
        message = json.loads(body.decode('utf-8'))
        event_type = message['event_type']
        data = message['data']
        timestamp = message['timestamp']
        
        print(f"\n📩 [{queue_name}] 接收事件:")
        print(f"   🕗 类型: {event_type}")
        print(f"   📅 时间: {timestamp}")
        print(f"   📝 数据: {json.dumps(data, ensure_ascii=False, indent=4)}")
        
        # 根据队列处理不同的业务
        if queue_name == 'order_processing':
            print(f"   ➡️ 进行订单处理...")
        elif queue_name == 'inventory':
            print(f"   ➡️ 更新库存...")
        elif queue_name == 'notification':
            print(f"   ➡️ 发送通知...")
        elif queue_name == 'analytics':
            print(f"   ➡️ 记录分析数据...")
        
        ch.basic_ack(delivery_tag=method.delivery_tag)
    
    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback
    )
    
    print(f"🔍 {queue_name} 消费者已启动...")
    channel.start_consuming()

# 注意：要运行消费者，请先发送消息再运行
# shopping_consumer('notification')

---

<a id='9-最佳实践'></a>
## 9. ⭐ 最佳实践

### 📋 安全性最佳实践

In [ ]:
best_practices = """
👕 最佳实践指单

1. 🔍 连接管理
   - 使用连接池
   - 实现连接重用
   - 处理连接断开

2. 📦 消息指针
   - 使用delivery_mode=2实现持久化
   - 手动确认而非auto_ack
   - 重试机制

3. 📅 队列设计
   - 队列声明为durable=True
   - 合理设计路由规则
   - 避免队列过度扩展

4. 💳 交易安全
   - 使用事务功能
   - 确保消息扭转
   - 监控消息死信队列

5. 🗓 监控和维护
   - 监控队列长度
   - 定期清理压除队列
   - 记录日志
"""

print(best_practices)

### 🔍 完全的连接管理示例

In [ ]:
import contextlib

class RobustConnection:
    """
    完全的RabbitMQ连接管理，支持重连
    """
    
    def __init__(
        self,
        host: str = 'localhost',
        port: int = 5672,
        username: str = 'guest',
        password: str = 'guest',
        max_retries: int = 3,
        retry_delay: int = 5
    ):
        self.host = host
        self.port = port
        self.username = username
        self.password = password
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self._connection = None
        self._channel = None
    
    def connect(self):
        """帝尹连接，支持重连
        """
        for attempt in range(self.max_retries):
            try:
                credentials = pika.PlainCredentials(self.username, self.password)
                parameters = pika.ConnectionParameters(
                    host=self.host,
                    port=self.port,
                    credentials=credentials,
                    heartbeat=600,
                    blocked_connection_timeout=300
                )
                self._connection = pika.BlockingConnection(parameters)
                self._channel = self._connection.channel()
                print(f"🟡 成功连接（第 {attempt + 1} 次尝试）")
                return self._channel
            except Exception as e:
                print(f"⚠️ 连接失败（第 {attempt + 1} 次）: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(self.retry_delay)
        raise Exception("连接超时")
    
    def close(self):
        """安全关闭连接
        """
        if self._connection and self._connection.is_open:
            self._connection.close()
            print("🔍 连接已关闭")
    
    @contextlib.contextmanager
    def session(self):
        """
        使用上下文管理连接
        """
        try:
            self.connect()
            yield self._channel
        finally:
            self.close()

# 测试完全连接管理
print("=" * 50)
print("📢 测试完全连接管理")
print("=" * 50)

try:
    robust_conn = RobustConnection()
    with robust_conn.session() as channel:
        channel.queue_declare(queue='robust_test', durable=True)
        print("🏆 使用上下文管理完成!")
except Exception as e:
    print(f"❌ 连接失败: {e}")

---

## 📝 学习结束

🏆 恭喜！您已完成了RabbitMQ交互式教程的学习！

### 📚 学习内容总结

| 章节 | 关键点 |
|------|------|
| 消息队列基础 | 解耐解耦、异步处理、制峰填谜 |
| 连接管理 | 连接池、重连机制、安全关闭 |
| 工作队列 | 负载均衡、劳动者轮询 |
| 交换机类型 | direct、fanout、topic、headers |
| 路由和绑定 | 路由键、通配符 |
| 实践应用 | 销售系统消息处理 |

### 💡 下一步学习建议

1. 🔍 **实际操作** - 启动RabbitMQ进行实际测试
2. 📅 **读取文档** - RabbitMQ官方文档
3. 📚 **进阶专题** - 事务、消息策略、集群模式

### 👀 有用的资源

- 📚 [RabbitMQ官方网站](https://www.rabbitmq.com/)
- 📝 [pika库文档](https://pika.readthedocs.io/)
- 🌐 [RabbitMQ教程](https://www.rabbitmq.com/tutorials/)

---

*📖 本教程由 MiniMax Agent 创佟*